In [ ]:



import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score

from xgboost import XGBClassifier


In [18]:
# LOAD DATA
df = pd.read_csv("email.csv")

print(df.head())
print(df.shape)
print(df.info())


  Category                                            Message
0      ham  Go until jurong point, crazy.. Available only ...
1      ham                      Ok lar... Joking wif u oni...
2     spam  Free entry in 2 a wkly comp to win FA Cup fina...
3      ham  U dun say so early hor... U c already then say...
4      ham  Nah I don't think he goes to usf, he lives aro...
(5573, 2)
<class 'pandas.DataFrame'>
RangeIndex: 5573 entries, 0 to 5572
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   Category  5573 non-null   str  
 1   Message   5573 non-null   str  
dtypes: str(2)
memory usage: 87.2 KB
None


In [19]:
# CLEAN DATA
# Keep only valid categories
df = df[df["Category"].isin(["ham", "spam"])]

# Remove missing values
df = df.dropna()

# Convert target into numbers
# ham  = 0
# spam = 1
df["Category"] = df["Category"].map({
    "ham": 0,
    "spam": 1
})

X = df["Message"]
y = df["Category"]

print(y.value_counts())

Category
0    4825
1     747
Name: count, dtype: int64


In [20]:
#  TRAIN TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


In [21]:
# TF-IDF
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_features=5000
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("TF-IDF shape:", X_train_tfidf.shape)


TF-IDF shape: (4457, 5000)


In [22]:
# 6. NEURAL NETWORK
#    ReLU + Adam
neurons = [16, 32, 64, 128]

nn_results = []

for n in neurons:

    model = MLPClassifier(
        hidden_layer_sizes=(n,),
        activation="relu",
        solver="adam",
        max_iter=50,
        early_stopping=True,
        random_state=42
    )

    model.fit(X_train_tfidf, y_train)

    pred = model.predict(X_test_tfidf)

    accuracy = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred)

    nn_results.append([
        n,
        accuracy,
        f1
    ])
    print(f"Neurons: {n}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print()


Neurons: 16
Accuracy: 0.9776
F1 Score: 0.9091

Neurons: 32
Accuracy: 0.9794
F1 Score: 0.9181

Neurons: 64
Accuracy: 0.9776
F1 Score: 0.9104

Neurons: 128
Accuracy: 0.9767
F1 Score: 0.9065



In [23]:
# NEURAL NETWORK RESULTS
nn_df = pd.DataFrame(
    nn_results,
    columns=["Neurons", "Accuracy", "F1 Score"]
)

print("Neural Network Results")
print(nn_df)


Neural Network Results
   Neurons  Accuracy  F1 Score
0       16  0.977578  0.909091
1       32  0.979372  0.918149
2       64  0.977578  0.910394
3      128  0.976682  0.906475


In [24]:
#  RANDOM FOREST
rf = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_tfidf, y_train)

rf_pred = rf.predict(X_test_tfidf)

rf_accuracy = accuracy_score(y_test, rf_pred)
rf_f1 = f1_score(y_test, rf_pred)

print("\nRandom Forest")
print("Accuracy:", rf_accuracy)
print("F1 Score:", rf_f1)

print(classification_report(
    y_test,
    rf_pred,
    target_names=["ham", "spam"]
))



Random Forest
Accuracy: 0.9838565022421525
F1 Score: 0.9366197183098591
              precision    recall  f1-score   support

         ham       0.98      1.00      0.99       966
        spam       0.99      0.89      0.94       149

    accuracy                           0.98      1115
   macro avg       0.98      0.95      0.96      1115
weighted avg       0.98      0.98      0.98      1115



In [25]:
# XGBOOST
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb.fit(X_train_tfidf, y_train)

xgb_pred = xgb.predict(X_test_tfidf)

xgb_accuracy = accuracy_score(y_test, xgb_pred)
xgb_f1 = f1_score(y_test, xgb_pred)

print("\nXGBoost")
print("Accuracy:", xgb_accuracy)
print("F1 Score:", xgb_f1)

print(classification_report(
    y_test,
    xgb_pred,
    target_names=["ham", "spam"]
))



XGBoost
Accuracy: 0.9668161434977578
F1 Score: 0.8664259927797834
              precision    recall  f1-score   support

         ham       0.97      0.99      0.98       966
        spam       0.94      0.81      0.87       149

    accuracy                           0.97      1115
   macro avg       0.95      0.90      0.92      1115
weighted avg       0.97      0.97      0.97      1115



In [26]:
# FINAL COMPARISON
best_nn = nn_df.loc[nn_df["F1 Score"].idxmax()]

comparison = pd.DataFrame({
    "Model": [
        f"Neural Network ({int(best_nn['Neurons'])} neurons)",
        "Random Forest",
        "XGBoost"
    ],

    "Accuracy": [
        best_nn["Accuracy"],
        rf_accuracy,
        xgb_accuracy
    ],

    "F1 Score": [
        best_nn["F1 Score"],
        rf_f1,
        xgb_f1
    ]
})

print("\nFinal Model Comparison")
print(comparison.sort_values(
    "F1 Score",
    ascending=False
))


Final Model Comparison
                         Model  Accuracy  F1 Score
1                Random Forest  0.983857  0.936620
0  Neural Network (32 neurons)  0.979372  0.918149
2                      XGBoost  0.966816  0.866426
